# 02.2 — Embeddings and search

Notebook 1 left us with 71 chunks. Now we make them searchable.

Keyword search would find chunks containing the words you typed. That breaks
the moment someone asks a question using different words than the document
uses — and people always do. Embeddings solve that by turning text into
numbers positioned so that similar meanings land near each other.

By the end you'll have a working retriever, and you'll have watched it return
the wrong thing with high confidence.

In [1]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [2]:
!pip install -q pymupdf4llm sentence-transformers ipywidgets

The install above pulls PyTorch, which is a large download the first time —
several hundred megabytes at least. It's cached afterwards.

If you're on a slow connection and have no GPU, the CPU-only build is much
smaller. Run this **before** the cell above:

```
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
```

Order matters — otherwise `sentence-transformers` fetches the CUDA build first.

## Rebuild the chunks

Same code as notebook 1, compressed into one cell. Each notebook stands on its
own, so you can open any of them without running the others first.

In [3]:
from pathlib import Path
import numpy as np
import pymupdf4llm

CORPUS = Path('../../corpus/docs')

NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]


def chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks = []
for name in NATIVE_PDFS:
    text = pymupdf4llm.to_markdown(str(CORPUS / name))
    for i, body in enumerate(chunk(text)):
        chunks.append({'doc': name, 'n': i, 'text': body})

print(f'{len(chunks)} chunks')

71 chunks


## Load the embedding model

`bge-small-en-v1.5` is small — about 33 million parameters — and runs fine on a
laptop CPU. It runs locally, so embedding is free and unlimited. That matters
more than it sounds: module 04 has you re-embed this corpus a dozen times to
compare chunking strategies, and you won't do that if each run costs money.

First run downloads the weights. After that it's cached.

In [4]:
from sentence_transformers import SentenceTransformer

MODEL = 'BAAI/bge-small-en-v1.5'
model = SentenceTransformer(MODEL)

print('dimensions:', model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dimensions: 384


384 dimensions. Every chunk, however long, becomes a list of 384 numbers.

That's the whole trick and also the whole problem — a 500-character chunk is
compressed into 384 floats, and everything that doesn't survive that
compression is gone. Module 05 is about what gets lost.

In [5]:
vectors = model.encode(
    [c['text'] for c in chunks],
    normalize_embeddings=True,
)

print('matrix:', vectors.shape)

matrix: (71, 384)


`normalize_embeddings=True` scales every vector to length 1. That's a
convenience: once vectors are unit length, the dot product between two of them
*is* their cosine similarity, so we can compare with a single matrix multiply
instead of a loop.

## Search

Embed the question the same way, then measure how close it lands to each chunk.
Highest scores win.

This is brute force — every query compares against all 71 vectors. Fine here,
hopeless at a million. Module 05 introduces a vector database to fix that.

In [6]:
def search(query, k=3):
    qv = model.encode([query], normalize_embeddings=True)[0]
    scores = vectors @ qv
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), chunks[i]) for i in top]


def show(query, k=3):
    print(f'QUERY: {query}\n')
    for score, c in search(query, k):
        print(f'  {score:.3f}  {c["doc"]} #{c["n"]}')
        print(f'         {c["text"][:130].strip()}...\n')

### It works

In [7]:
show('What is the maximum emergency procurement without competitive sourcing?')

QUERY: What is the maximum emergency procurement without competitive sourcing?

  0.798  sahel-procurement-policy-v3.pdf #4
         5,000,000 without competitive sourcing. Emergency procurement must be reported to the Management Procurement Committee at its next...

  0.691  sahel-procurement-policy-v3.pdf #1
         parately by the Group Technology Procurement Standard. 

## **2. Approval thresholds** 

|**Value (NGN)**|**Sourcing requirement**...

  0.659  sahel-procurement-policy-v3.pdf #2
         75,000,000|Competitive bid with technical and financial evaluation|Board Finance and General Purpose Committee|



Splitting a re...



Note that the question says *maximum* and *without competitive sourcing*, while
the document says *up to* and *without competitive sourcing*. Keyword search
would have struggled. This is what embeddings buy you.

Try a couple of your own before moving on. Questions from
`corpus/golden_questions.csv` are a good source.

In [8]:
show('How long do I have to submit an expense claim?')

QUERY: How long do I have to submit an expense claim?

  0.771  sahel-employee-handbook-2023.pdf #8
         f the divisional head and is booked in economy class for all grades below General Manager. 

Expense claims must be submitted with...

  0.767  sahel-employee-handbook-2025.pdf #8
         the divisional head and is booked in economy class for all grades below General Manager. 

Expense claims must be submitted withi...

  0.667  sahel-procurement-policy-v3.pdf #6
         . Advance payment exceeding thirty per cent of contract value requires an advance payment guarantee from a bank acceptable to the...



## Now break it

Three queries below. Each one fails, and each failure is a module later in the
course. Read the results carefully rather than skimming — the point is to
notice what's wrong before being told.

### One: the same question, two answers

In [9]:
show('How many days of annual leave do confirmed staff get?', k=4)

QUERY: How many days of annual leave do confirmed staff get?

  0.791  sahel-employee-handbook-2023.pdf #5
         ously. 

Staff at Assistant Manager grade and above are entitled to an additional three working days. 

## **5. Other leave** 

**...

  0.790  sahel-employee-handbook-2025.pdf #5
         eously. 

Staff at Assistant Manager grade and above are entitled to an additional five working days. 

## **5. Other leave** 

**...

  0.785  sahel-employee-handbook-2023.pdf #4
         endar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 5...

  0.782  sahel-employee-handbook-2025.pdf #4
         endar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 1...



Look at which documents came back.

The corpus contains the employee handbook twice — a 2023 edition and a 2025
edition. The rules changed between them. Both are in the index, both are about
annual leave, and the retriever has no idea one supersedes the other.

If the 2023 chunk ranks higher, an LLM reading it will answer confidently, cite
a real document, and be wrong. There's no hallucination involved — the system
faithfully reported a source that shouldn't have been there.

That failure has a name in this course: **faithful incorrectness**. It's the
most common serious bug in production RAG, and it isn't a retrieval bug. It's
an ingestion bug, because nothing recorded that one document replaced the
other. Module 03.

### Two: exact identifiers

In [10]:
show('What does circular NFSC/CIR/2025/02 change?')

QUERY: What does circular NFSC/CIR/2025/02 change?

  0.865  nfsc-circular-2025-02-amendment.pdf #0
         Circular NFSC/CIR/2025/02 

# **National Financial Services Commission** 

To: All Licensed Microfinance Banks, Payment Service Pr...

  0.834  nfsc-circular-2024-07-cybersecurity.pdf #0
         Circular NFSC/CIR/2024/07 

# **National Financial Services Commission** 

To: All Licensed Microfinance Banks, Payment Service Pr...

  0.730  nfsc-circular-2025-02-amendment.pdf #2
         ns with total assets below NGN 5 billion is brought forward from 31 December 2025 to **30 September 2025** . All other dates in th...



Did the right circular come first?

`NFSC/CIR/2025/02` and `NFSC/CIR/2024/07` are nearly identical as strings, and
embeddings are built to collapse small differences in wording. A code where one
digit changes the meaning entirely is exactly the wrong shape for that.

The same problem hits product codes, invoice numbers, and names. The fix is not
a better embedding model — it's keyword search running alongside this one,
which is module 07.

### Three: the missing answer

In [ ]:
show('What was revenue in 2020?')

Nothing useful, and the scores are low across the board.

The annual report contains that figure, but only inside a chart image. The
text says revenue "rose substantially" and never gives a number. There is no
chunk that can answer this, so the retriever returns its least-bad guesses
instead of saying so.

A system that can't tell the difference between "here is the answer" and
"here is the closest thing I found" will hand an LLM irrelevant context and let
it improvise. Recognising this case is module 09; reading the chart is module
15.

## What's next

You have a retriever. It answers some questions well and fails on others in
three distinct ways.

Notebook 3 sends the retrieved chunks to an LLM and gets an actual answer out.
Then — and this is the important part — notebook 5 stops us judging any of this
by eye and puts a number on it.

Resist fixing anything. Every failure above has a module, and fixing them
before you can measure them means never finding out whether the fix helped.